# Imports

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split
from torchvision import transforms as T
from sklearn.utils.class_weight import compute_class_weight
import wandb
import optuna
from sklearn.metrics import f1_score

# Convolutive Neural Network Structure

In [10]:
class CNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=3,
                 normalize2d=nn.BatchNorm2d, normalize1d=nn.BatchNorm1d,
                 pool=nn.MaxPool2d,
                 drop_conv_prob=0.2, drop_fc_prob=0.4,
                 activation=nn.ReLU()):
        
        super(CNN, self).__init__()

        self.activation = activation
        
        self.conv1 = nn.Conv2d(in_channels, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.bn1 = normalize2d(32)
        self.conv1_2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.bn1_2 = normalize2d(32)
        
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.bn2 = normalize2d(64)
        self.conv2_2 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.bn2_2 = normalize2d(64)
        
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.bn3 = normalize2d(128)
        self.conv3_2 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.bn3_2 = normalize2d(128)

        self.dropout2d = nn.Dropout2d(p=drop_conv_prob)

        self.pool = pool(kernel_size=2, stride=2)
        
        self.fc1 = nn.Linear(128 * 28 * 28, 128)
        self.bn_fc1 = normalize1d(128)

        self.dropout = nn.Dropout(p=drop_fc_prob)

        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.activation(x)
        x = self.dropout2d(x)

        x = self.conv1_2(x)
        x = self.bn1_2(x)
        x = self.activation(x)
        x = self.pool(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.activation(x)
        x = self.dropout2d(x)

        x = self.conv2_2(x)
        x = self.bn2_2(x)
        x = self.activation(x)
        x = self.pool(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.activation(x)
        x = self.dropout2d(x)

        x = self.conv3_2(x)
        x = self.bn3_2(x)
        x = self.activation(x)
        x = self.pool(x)

        x = x.reshape(x.shape[0], -1)

        x = self.fc1(x)
        x = self.bn_fc1(x)
        x = self.activation(x)
        x = self.dropout(x)

        x = self.fc2(x)

        return x

# Dataset Loading and Augmentation

In [11]:
class ImageDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        image = Image.open(img_path).convert('L')
        
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)
            
        return image, label

val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5])
])

train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(30),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5])
])

data_dir = Path('Breast-Cancer-Dataset')
classes = [d.name for d in data_dir.iterdir() if d.is_dir()]
label_map = {name: i for i, name in enumerate(classes)}
num_classes = len(classes)

print(f"Clases encontradas: {label_map}")

file_paths = []
labels = []
for class_name, label_idx in label_map.items():
    class_dir = data_dir / class_name
    for img_path in class_dir.glob('*.[jp][pn]g'): 
        file_paths.append(str(img_path))
        labels.append(label_idx)

print(f"Total de imágenes encontradas: {len(file_paths)}")

train_paths, val_paths, train_labels, val_labels = train_test_split(
    file_paths, 
    labels, 
    test_size=0.2,
    random_state=42,
    stratify=labels
)

train_dataset = ImageDataset(train_paths, train_labels, transform=train_transform)
val_dataset = ImageDataset(val_paths, val_labels, transform=val_transform)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Imágenes de entreno: {len(train_dataset)}")
print(f"Imágenes de validación: {len(val_dataset)}")


Clases encontradas: {'benign': 0, 'malignant': 1, 'normal': 2}
Total de imágenes encontradas: 780
Imágenes de entreno: 624
Imágenes de validación: 156


# Training and Validation

In [ ]:
wandb.login(key="")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Juan Diego\_netrc


True

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)

torch.manual_seed(42)
np.random.seed(42)

Usando dispositivo: cpu


In [14]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

In [15]:

OPTIMIZERS = {
    "AdamW": torch.optim.AdamW,
    "Adam": torch.optim.Adam,
    "NAdam": torch.optim.NAdam,
    "SGD": torch.optim.SGD,
}

ACTIVACTIONS = {
    "ReLU": nn.ReLU(),
    "LeakyReLU": nn.LeakyReLU(),
}

NORMALIZATIONS_2D = {
    "BatchNorm2d": nn.BatchNorm2d,
    "InstanceNorm2d": nn.InstanceNorm2d,
}

NORMALIZATIONS_1D = {
    "BatchNorm1d": nn.BatchNorm1d,
    "InstanceNorm1d": nn.InstanceNorm1d,}

POOLS = {
    "MaxPool2d": nn.MaxPool2d,
    "AvgPool2d": nn.AvgPool2d,
}


In [ ]:
# ====================== OPTUNA'S OBJECTIVE ======================
def objective(trial):

    # ------------------- HYPERPARAMETERS TO OPTIMIZE -------------------
    lr = trial.suggest_float("learning_rate", 0.0001, 0.005, log=True)
    drop_fc = trial.suggest_float("dropout", 0.2, 0.8)
    drop_conv = trial.suggest_float("dropout_conv", 0.0, 0.4)
    weight_decay = trial.suggest_float("weight_decay", 0.000001, 0.001, log=True)
    patience = trial.suggest_int("patience", 5, 12)
    opti = trial.suggest_categorical("optimizer", list(OPTIMIZERS.keys()))
    activ_func = trial.suggest_categorical("activ_func", list(ACTIVACTIONS.keys()))
    normalize_2d = trial.suggest_categorical("normalize_2d", list(NORMALIZATIONS_2D.keys()))
    normalize_1d = trial.suggest_categorical("normalize_1d", list(NORMALIZATIONS_1D.keys()))
    pools = trial.suggest_categorical("pool", list(POOLS.keys()))

    # ------------------- INITIALIZE WANDB FOR TRIAL -------------------
    wandb.init(
        project="BUSI-CNN-OPTUNA-F1",
        name=f"trial-{trial.number}",
        config={
            "learning_rate": lr,
            "weight_decay": weight_decay,
            "patience": patience,
            "epochs": 50,
            "optimizer": opti,
            "activation_function": activ_func,
            "normalize_2d": normalize_2d,
            "normalize_1d": normalize_1d,
            "pool": pools,
            "dropout": drop_fc,
            "dropout_conv": drop_conv,
        }
    )
    config = wandb.config

    # ------------------- MODEL -------------------
    model = CNN(
        in_channels=1,
        num_classes=3,
        normalize2d=NORMALIZATIONS_2D[normalize_2d],
        normalize1d=NORMALIZATIONS_1D[normalize_1d],
        pool=POOLS[pools],
        activation=ACTIVACTIONS[activ_func],
        drop_conv_prob=drop_conv,
        drop_fc_prob=drop_fc
        ).to(device)

    # ------------------- OPTIMIZER -------------------

    if opti == "SGD":
        optimizer = OPTIMIZERS[opti](
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay,
            momentum=0.9
        )

    else:
        optimizer = OPTIMIZERS[opti](
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    # ------------------- LOSS FUNCTION WITH CLASS WEIGHTS -------------------
    criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

    # ------------------- LEARNING RATE SCHEDULER -------------------
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.1,
        patience=patience,
    )

    # ------------------- TRAINING -------------------
    best_val_loss = float("inf")
    patience_counter = 0
    NUM_EPOCHS = config.epochs

    for epoch in range(NUM_EPOCHS):

        # -------- TRAIN --------
        model.train()
        train_loss_sum, correct_train, n_train = 0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * images.size(0)
            _, pred = outputs.max(1)
            n_train += labels.size(0)
            correct_train += (pred == labels).sum().item()

        train_loss = train_loss_sum / n_train
        train_acc = 100 * correct_train / n_train

        # -------- VAL --------
        model.eval()
        val_loss_sum, correct_val, n_val = 0, 0, 0

        all_preds, all_labels = [], []

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss_sum += loss.item() * images.size(0)
                _, pred = outputs.max(1)
                n_val += labels.size(0)
                correct_val += (pred == labels).sum().item()

                all_preds.extend(pred.cpu().numpy())
                all_labels.extend(labels.cpu().numpy()) 

        val_loss = val_loss_sum / n_val
        val_acc = 100 * correct_val / n_val

        f1 = f1_score(all_labels, all_preds, average='weighted')

        scheduler.step(val_loss)

        trial.report(val_loss, epoch)

        wandb.log({
            "epoch": epoch,
            "train/loss": train_loss,
            "train/acc": train_acc,
            "val/loss": val_loss,
            "val/acc": val_acc,
            "val/f1_score": f1,
            "optimizer": opti,
            "lr": optimizer.param_groups[0]['lr']
        })

        # -------- EARLY STOPPING --------
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    # =================== CONFUSION MATRIX  ===================
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = outputs.max(1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    wandb.log({
        "confusion_matrix": wandb.plot.confusion_matrix(
            y_true=all_labels,
            preds=all_preds,
            class_names=["benign", "malignant", "normal"]
        )
    })
    
    wandb.finish()
    return f1


# ====================== OPTUNA'S STUDY ======================
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)

print("Best trial:", study.best_trial.params)

[I 2025-12-08 23:36:50,858] A new study created in memory with name: no-name-c4f324e1-78c0-48e0-ada2-b07cc160095d


c:\Users\Juan Diego\anaconda3\Lib\site-packages\torch\nn\modules\instancenorm.py:115: UserWarning: input's size at dim=0 does not match num_features. You can silence this warning by not passing in num_features, which is not used because affine=False
  warnings.warn(


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▂▃▃▃▄▅▄▅▄▅▅▄▅▅▆▅▅▆▆▆▆▆▇▇▇▇▆█▇▇▇▇▇▇▇▇▇██
train/loss,█▇▇▆▆▅▅▆▅▅▅▅▅▅▄▄▄▄▄▄▄▄▃▃▃▂▃▃▃▂▂▃▂▂▂▂▂▂▂▁
val/acc,▁▂▂▂▄▄▅▆▃▅▆▆▇▇▇▇▇▇▇█▇▇▇▆▇▇▇█▇▇█▇▇▇▇▇█▇▇█
val/loss,███▇▆▆▅▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁
epoch,49
lr,0.00026
optimizer,NAdam
train/acc,64.58333
train/loss,0.7557


[I 2025-12-09 00:59:19,342] Trial 0 finished with value: 0.6595636713199127 and parameters: {'learning_rate': 0.0002613318377399962, 'dropout': 0.6946167302317947, 'dropout_conv': 0.3634866832929872, 'weight_decay': 0.0007227811438839358, 'patience': 6, 'optimizer': 'NAdam', 'activ_func': 'LeakyReLU', 'normalize_2d': 'BatchNorm2d', 'normalize_1d': 'InstanceNorm1d', 'pool': 'MaxPool2d'}. Best is trial 0 with value: 0.6595636713199127.


epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇███
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▂▄▃▃▄▄▄▅▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇█████
train/loss,█▇▇▇▇▆▆▆▅▅▅▅▅▄▅▅▄▄▃▄▃▃▃▂▂▂▂▂▂▂▁▁▁
val/acc,▄▂▁▃▄▄▄▂▃▅▇▇▆▆▆▆▇▇███▇▇▇▆▇█▇▇█▇▇▇
val/loss,██▇▇▆▆▅▅▄▄▄▃▃▃▂▃▂▂▂▂▂▂▁▂▁▂▂▃▂▂▃▂▅
epoch,32
lr,0.00022
optimizer,Adam
train/acc,76.92308
train/loss,0.55609


...

epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▁▁▂▃▃▃▃▄▃▄▄▄▄▄▅▄▅▅▅▅▅▆▇▆▇▇▆▇▇▇▇▆█▇▇████
train/loss,█▆▆▆▆▆▅▅▅▅▅▄▄▅▄▄▄▄▃▄▄▃▃▂▂▂▂▃▂▂▂▂▂▁▂▁▁▁▁▁
val/acc,▁▃▄▃▃▄▄▅▄▅▅▄▃▅▆▄▅▃▆▅▆▆▇▇▆▇▇▇▇▇█▆▆▇▇▆▇▇▇▇
val/loss,█▆▅▅▅▄▅▅▄▄▄▄▃▃▄▄▃▃▃▃▃▂▁▂▂▁▃▂▂▁▂▁▂▂▃▁▃▂▁▂
epoch,49
lr,0.00489
optimizer,SGD
train/acc,78.6859
train/loss,0.50474


[I 2025-12-09 09:49:12,880] Trial 8 finished with value: 0.6476315565598316 and parameters: {'learning_rate': 0.004885865478139291, 'dropout': 0.40449565406209687, 'dropout_conv': 0.13853563755210888, 'weight_decay': 0.00010162681188603044, 'patience': 10, 'optimizer': 'SGD', 'activ_func': 'ReLU', 'normalize_2d': 'BatchNorm2d', 'normalize_1d': 'BatchNorm1d', 'pool': 'AvgPool2d'}. Best is trial 2 with value: 0.5284741398615714.


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▂▂▂▂▃▄▄▄▄▄▄▅▅▅▇▅▅▆▅▆▆▇▅▆▆▇▇▆▇▇▇▆▇▇▇▇█▆█
train/loss,██▇▇▇▆▆▆▆▆▆▅▅▅▅▄▅▄▄▄▄▃▂▃▃▄▃▃▂▃▂▂▂▂▂▂▁▁▂▁
val/acc,▁▄▅▄▅▅▆▆▇▆▆▇▇▆▇▆▇▇▇▇█▇▇█▇▇▇█▇▇▇▇▇██▇▇█▇▇
val/loss,█▆▆▆▆▅▅▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▃▁▁▁▁▂▁▂▂▂▂
epoch,49
lr,0.00152
optimizer,AdamW
train/acc,66.82692
train/loss,0.74005


[I 2025-12-09 11:14:29,657] Trial 9 finished with value: 0.7084050285510528 and parameters: {'learning_rate': 0.0015201720476041547, 'dropout': 0.42970528686337406, 'dropout_conv': 0.37468146420138476, 'weight_decay': 0.0002520745658464841, 'patience': 12, 'optimizer': 'AdamW', 'activ_func': 'ReLU', 'normalize_2d': 'BatchNorm2d', 'normalize_1d': 'InstanceNorm1d', 'pool': 'MaxPool2d'}. Best is trial 2 with value: 0.5284741398615714.


Best trial: {'learning_rate': 0.0021905344042955893, 'dropout': 0.49372640534661233, 'dropout_conv': 0.09238555523041941, 'weight_decay': 0.000604584451597831, 'patience': 8, 'optimizer': 'NAdam', 'activ_func': 'ReLU', 'normalize_2d': 'BatchNorm2d', 'normalize_1d': 'InstanceNorm1d', 'pool': 'MaxPool2d'}
